In [6]:
import sqlite3
import numpy as np
import pandas as pd
import xarray as xr
import os
import sys

sys.path.append(os.path.abspath('../../'))
from src.hydro_utils import calculate_grid_cell_areas

In [13]:
mask_path = "/Users/ljob/Desktop/cnbs-predictor/data/input/SFS_GL_mask.nc"
database_path = "/Users/ljob/Desktop/Data/SFS_reforecast/sfs_reforecast_data_runoff.db"

variable_name = "watrsfc"
long_name = "runoff"
months = ["03", "04", "05", "06", "07", "08", "09", "11"]

start_date = "1991-03-01"
end_date = "2026-03-01"

In [8]:
mask_ds = xr.open_dataset(mask_path)

mask_variables = [
    var
    for var in mask_ds.data_vars
    #if var.endswith("_lake") or var.endswith("_land")
    if var.endswith("_land")
]

print("Processing masks:")
for mask_var in mask_variables:
    print(f"  {mask_var}")

mask_lat = mask_ds["lat"].values
mask_lon = mask_ds["lon"].values

Processing masks:
  superior_land
  michigan-huron_land
  erie_land
  ontario_land


In [9]:
# Calculate grid-cell area for area-weighted lake/land averages.
area = calculate_grid_cell_areas(mask_lon, mask_lat)

In [14]:
print()
print("Opening database...")

conn = sqlite3.connect(database_path)

cursor = conn.cursor()

# Create the table if it doesn't exist
cursor.execute(
    """
    CREATE TABLE IF NOT EXISTS reforecast_data (
        init_time TEXT,
        member INTEGER,
        lead INTEGER,
        valid_time TEXT,
        lake TEXT,
        surface_type TEXT,
        component TEXT,
        value REAL
    )
    """
)

# Create Index
cursor.execute(
    """
    CREATE INDEX IF NOT EXISTS idx_reforecast_lookup
    ON reforecast_data (
        init_time,
        member,
        lead,
        lake,
        surface_type,
        component
    )
    """
)

conn.commit()


Opening database...


In [ ]:
for month in months:

    zarr_path = f"s3://noaa-oar-sfsdev-pds/experiments/beta1/reforecast/{month}/atm_monthly.zarr"

    try:
        ds = xr.open_zarr(
        zarr_path,
        storage_options={"anon": True}
        )

        variable = ds[variable_name]

        for init_time in variable.init.sel(init=slice(start_date, end_date)).values:

            # -----------------------------------------------------------
            # Convert initialization time to pandas Timestamp
            # -----------------------------------------------------------

            init_timestamp = pd.Timestamp(init_time)
            seconds_in_month = init_timestamp.days_in_month * 86400

            print()
            print("=" * 70)
            print(
                f"Processing initialization: "
                f"{init_timestamp:%Y-%m-%d}"
            )
            print("=" * 70)

            for member in variable.member.values:

                member = int(member)

                print()
                print(f"  Member: {member}")
                for lead in variable.lead.values:

                    lead = int(lead)
                    valid_time = (
                        init_timestamp
                        + pd.DateOffset(months=lead)
                    )


                    print(
                        f"    Lead {lead:02d} "
                        f"-> {valid_time:%Y-%m}"
                    )

                    # ===================================================
                    # Select temperature field
                    # ===================================================

                    runoff = variable.sel(
                        init=init_time,
                        member=member,
                        lead=lead,
                    )

                    # ===================================================
                    # Cut temperature field to mask extent
                    # ===================================================

                    runoff_cut = runoff.sel(
                        lat=slice(
                            mask_lat.min(),
                            mask_lat.max(),
                        ),
                        lon=slice(
                            mask_lon.min(),
                            mask_lon.max(),
                        ),
                    )

                    runoff_values = runoff_cut.values

                    # ===================================================
                    # Loop through mask variables
                    # ===================================================

                    for mask_var in mask_variables:

                        lake, surface_type = mask_var.rsplit(
                            "_",
                            1,
                        )

                        # ------------------------------------------------
                        # Get fractional mask
                        # ------------------------------------------------

                        mask = mask_ds[mask_var].values

                        runoff_avg = (
                            np.nansum(runoff_values * mask * area)
                            / np.nansum(mask * area)
                        )
                        print(
                            f"      {lake} "
                            f"-> {long_name}: {runoff_avg:.4f}"
                        )
                        
                        # ------------------------------------------------
                        # Insert result into database
                        # ------------------------------------------------

                        cursor.execute(
                            """
                            INSERT INTO reforecast_data (
                                init_time,
                                member,
                                lead,
                                valid_time,
                                lake,
                                surface_type,
                                component,
                                value
                            )
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                            """,
                            (
                                init_timestamp.strftime("%Y-%m-%d"),
                                member,
                                lead,
                                valid_time.strftime("%Y-%m-%d"),
                                lake,
                                surface_type,
                                long_name,
                                float(runoff_avg)
                            ),
                        )

                conn.commit()

    except Exception as e:

        print()
        print("=" * 70)
        print(f"Error processing month {month}: {e}")
        print("=" * 70)
        print()

        continue

# ===============================================================
# Close database
# ===============================================================

conn.close()

print()
print("=" * 70)
print("Processing complete.")
print("=" * 70)
print()
print(f"Database: {database_path}")            



Processing initialization: 1991-03-01

  Member: 0
    Lead 00 -> 1991-03
      superior -> runoff: 0.1406
      michigan-huron -> runoff: 0.7380
      erie -> runoff: 0.3438
      ontario -> runoff: 0.7343
    Lead 01 -> 1991-04
      superior -> runoff: 0.6873
      michigan-huron -> runoff: 0.8331
      erie -> runoff: 0.4382
      ontario -> runoff: 0.8097
    Lead 02 -> 1991-05
      superior -> runoff: 0.7661
      michigan-huron -> runoff: 0.6607
      erie -> runoff: 0.2246
      ontario -> runoff: 0.5451
    Lead 03 -> 1991-06
      superior -> runoff: 0.4809
      michigan-huron -> runoff: 0.4881
      erie -> runoff: 0.1448
      ontario -> runoff: 0.2775
    Lead 04 -> 1991-07
      superior -> runoff: 0.5264
      michigan-huron -> runoff: 0.4021
      erie -> runoff: 0.1705
      ontario -> runoff: 0.3733
    Lead 05 -> 1991-08
      superior -> runoff: 0.3672
      michigan-huron -> runoff: 0.2605
      erie -> runoff: 0.1339
      ontario -> runoff: 0.2468
    Lead 06 